In [2]:
# 1. COLAB İÇİN OTOMATİK KÜTÜPHANE KURULUM HÜCRESİ
import sys

if 'google.colab' in sys.modules:
    print("🚀 Google Colab ortamı tespit edildi, gerekli kütüphaneler yükleniyor...")
    !pip install -q lightgbm requests
else:
    print("💻 Yerel ortam (VS Code) tespit edildi, yerel kütüphaneler kullanılıyor.")

🚀 Google Colab ortamı tespit edildi, gerekli kütüphaneler yükleniyor...


In [3]:
# 1. TÜM KÜTÜPHANELER VE KÜRESEL AYARLAR
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print("✔️ Tasarım matrisi için tüm algoritmalar ve kütüphaneler başarıyla yüklendi!")

✔️ Tasarım matrisi için tüm algoritmalar ve kütüphaneler başarıyla yüklendi!


In [ ]:
# 2. VERİ POPÜLASYON GRAFİKLERİ
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import re

def load_large_drive_csv(file_id):
    url = "https://docs.google.com/uc?export=download"
    session = requests.Session()

    # İlk istek: Google'ın güvenlik/uyarı sayfasını tetikliyoruz
    response = session.get(url, params={'id': file_id}, stream=True)

    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break

    if not token:
        match = re.search(r'confirm=([a-zA-Z0-9_-]+)', response.text)
        if match:
            token = match.group(1)

    if token:
        response = session.get(url, params={'id': file_id, 'confirm': token}, stream=True)

    if response.text.startswith('<!DOCTYPE html>') or '<html' in response.text[:200]:
        if "Access Denied" in response.text or "sign in" in response.text.lower():
            raise PermissionError("Google Drive bağlantı izni hatası! Lütfen paylaşım ayarlarını kontrol edin.")

    return pd.read_csv(io.BytesIO(response.content), sep=None, engine='python')

# 1. AŞAMA: Öznitelik verilerini Google Drive'dan indiriyoruz
FILE_ID = '1sC62oD6NdA9bpSlRV305Yyo4leDSlEbm'
print("⏳ 90.9 MB'lık EEG Öznitelik Matrisi Google Drive'dan güvenli hatla indiriliyor...")
df_temiz = load_large_drive_csv(FILE_ID)
print(f"✔️ Öznitelik matrisi başarıyla yüklendi! Satır/Sütun Boyutu: {df_temiz.shape}")

# 2. AŞAMA: Eksik olan etiketleri (Label) resmi OpenNeuro deposundan canlı çekip eşliyoruz
print("⏳ Klinik etiketler (participants.tsv) OpenNeuro ana sunucularından otomatik olarak çekiliyor...")
tsv_urls = [
    "https://raw.githubusercontent.com/OpenNeuroDatasets/ds004504/master/participants.tsv",
    "https://raw.githubusercontent.com/OpenNeuroDatasets/ds004504/main/participants.tsv"
]

participants_df = None
for url in tsv_urls:
    try:
        res = requests.get(url, timeout=15)
        if res.status_code == 200:
            participants_df = pd.read_csv(io.StringIO(res.text), sep='\t')
            print(f"📖 Etiket dökümanı buluttan başarıyla doğrulandı.")
            break
    except:
        continue

# Buluttan çekme başarısız olursa yerel yedek kontrolü (VS Code için alternatif hat)
if participants_df is None:
    try:
        participants_df = pd.read_csv('participants.tsv', sep='\t')
        print("📖 Etiket dökümanı yerel dizinden (participants.tsv) okundu.")
    except:
        raise FileNotFoundError("Kritik Hata: 'participants.tsv' dosyası ne internetten ne de yerel dizinden okunamadı!")

# 3. AŞAMA: ID bazlı akıllı eşleme döngüsü (Mapping)
etiket_sozlugu = dict(zip(participants_df['participant_id'].astype(str).str.strip(),
                          participants_df['Group'].astype(str).str.strip().str.upper()))

df_temiz['Label'] = df_temiz['Subject_ID'].astype(str).str.strip().map(etiket_sozlugu)
print("✔️ Klinik etiketler 'Subject_ID' üzerinden ana matrise başarıyla enjekte edildi!")

# Popülasyon Grafiklerinin Çizimi
hasta_bazli_grup = df_temiz.groupby('Subject_ID')['Label'].first()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. GRAFİK: HASTA BAZLI DAĞILIM
p_counts = hasta_bazli_grup.value_counts().reindex(['A', 'C', 'F'], fill_value=0)
axes[0].pie(p_counts, labels=['Alzheimer (A)', 'Control (C)', 'FTD (F)'], autopct='%1.1f%%',
            colors=['#ff9999','#66b3ff','#99ff99'], startangle=140, explode=(0.04, 0.04, 0.04))
axes[0].set_title("Hasta Bazlı Klinik Popülasyon Dağılımı (Toplam: 88 Hasta)", fontsize=13, weight='bold')

# 2. GRAFİK: EPOCH BAZLI DAĞILIM
e_counts = df_temiz['Label'].value_counts().reindex(['A', 'C', 'F'], fill_value=0)
sns.barplot(x=['Alzheimer (A)', 'Control (C)', 'FTD (F)'], y=e_counts.values, ax=axes[1], palette='Set2')
axes[1].set_title("Epoch (4 sn Sinyal Penceresi) Bazlı Satır Dağılımı", fontsize=13, weight='bold')

for i, v in enumerate(e_counts.values):
    axes[1].text(i, v + (max(e_counts.values) * 0.02), f"{v:,}", ha='center', weight='bold', fontsize=11)

plt.tight_layout()
plt.show()

⏳ 90.9 MB'lık EEG Öznitelik Matrisi Google Drive'dan güvenli hatla indiriliyor...


In [ ]:
# 3. ÖLÇEKLENMİŞ VE UÇ DEĞERLERİ GİZLENMİŞ ARTIK TABANA YAPIŞMAYAN THETA BOXPLOTU
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# COLAB UYUMLULUK DÜZELTMESİ: 'df' yerine bir önceki hücreden gelen 'df_temiz' matrisini kullanıyoruz
gamma_sutunlari = [col for col in df_temiz.columns if 'Gamma' in col]
df_temiz = df_temiz.drop(columns=gamma_sutunlari, errors='ignore')

# Etiketleri temizleyip eşliyoruz
y_clean = df_temiz['Label'].astype(str).str.strip().str.upper()

theta_sutunlari = [col for col in df_temiz.columns if 'Theta' in col and 'PSD' in col]
# Grafik ölçeğini kurtarmak ve 1e-9 sıkışmasını engellemek için 1e9 ile çarpıyoruz
df_temiz['Ortalama_Theta_Gucu'] = df_temiz[theta_sutunlari].mean(axis=1) * 1e9

sınıf_isimleri = {'A': 'Alzheimer (AD)', 'F': 'Frontotemporal (FTD)', 'C': 'Healthy (CN)'}
df_temiz['Klinik_Grup'] = y_clean.map(sınıf_isimleri)

plt.figure(figsize=(9, 5.5))
# showfliers=False ekleyerek ezilmeyi tamamen çözüyoruz
sns.boxplot(x='Klinik_Grup', y='Ortalama_Theta_Gucu', data=df_temiz,
            palette=['#66c2a5', '#fc8d62', '#8da0cb'], width=0.6,
            showfliers=False,
            order=['Alzheimer (AD)', 'Frontotemporal (FTD)', 'Healthy (CN)'])

plt.title("Gruplara Göre Ortalama Theta Gücü (EDA)", fontsize=13, weight='bold', pad=10)
plt.xlabel("", fontsize=11)
plt.ylabel("Ortalama Theta Gücü (x10^-9 V^2/Hz)", fontsize=11)
plt.grid(axis='y', linestyle='-', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 4. MULTI-MODEL VE MULTI-FEATURE BÜYÜK BİLEŞİK HESAPLAMA DÖNGÜSÜ
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# COLAB UYUMLULUK GÜVENCESİ: Bağımlı değişkeni doğrudan bu hücrede sağlama alıyoruz
y_clean = df_temiz['Label'].astype(str).str.strip().str.upper()

X_raw = df_temiz.drop(columns=['Subject_ID', 'Epoch_No', 'Label', 'Ortalama_Theta_Gucu', 'Klinik_Grup'], errors='ignore')
groups_all = df_temiz['Subject_ID']

# Özellik Kümeleri Ayrışımı
psd_sutunlari = [col for col in X_raw.columns if 'PSD' in col]
X_set_PSD = X_raw[psd_sutunlari]

# Öznitelik Seçimi (Feature Importance Hattı)
model_fi = lgb.LGBMClassifier(objective='multiclass', class_weight='balanced', random_state=42, n_estimators=100, verbosity=-1)
model_fi.fit(X_raw, y_clean.map({'C':0, 'A':1, 'F':2}))
top_40_features = pd.DataFrame({'F': X_raw.columns, 'I': model_fi.feature_importances_}).sort_values(by='I', ascending=False)['F'].head(40).tolist()
X_set_FUSION = X_raw[top_40_features]

scenarios = [
    ({'C': 0, 'A': 1}, "Senaryo 1: AD vs Control", ['Control', 'Alzheimer']),
    ({'C': 0, 'F': 1}, "Senaryo 2: FTD vs Control", ['Control', 'FTD']),
    ({'F': 0, 'A': 1}, "Senaryo 3: AD vs FTD", ['FTD', 'Alzheimer']),
    ({'C': 0, 'A': 1, 'F': 2}, "Senaryo 4: 3'lü Sınıflandırma", ['Control', 'Alzheimer', 'FTD'])
]

master_results = []
champion_confusion_matrices = {} # En iyi şampiyon modelin (LightGBM) matrisleri için

for label_mapping, title, class_names in scenarios:
    print(f"🔄 {title} için 6'lı kombinasyon matrisi LOSO ile hesaplanıyor...")
    target_classes = list(label_mapping.keys())
    is_multiclass = len(target_classes) > 2

    mask = y_clean.isin(target_classes)
    y_scen = y_clean[mask].map(label_mapping).reset_index(drop=True)
    groups_scen = groups_all[mask].reset_index(drop=True)

    setups = [
        ("Klasik Baseline (Sadece PSD)", X_set_PSD[mask].reset_index(drop=True)),
        ("Önerilen Çoklu Füzyon (PSD+CWT+EMD)", X_set_FUSION[mask].reset_index(drop=True))
    ]

    for setup_name, X_scen in setups:
        # 3 Farklı Jenerasyon Algoritma Tanımlaması (Regüle Edilmiş Yapı)
        models = {
            "LDA (Lineer Klasik)": LinearDiscriminantAnalysis(),
            "Random Forest (Geleneksel Topluluk)": RandomForestClassifier(class_weight='balanced', max_depth=8, random_state=42, n_jobs=-1),
            "LightGBM (Modern Boosting)": lgb.LGBMClassifier(objective='multiclass' if is_multiclass else 'binary',
                                                             class_weight='balanced', max_depth=5, learning_rate=0.05,
                                                             n_estimators=100, verbosity=-1, random_state=42)
        }

        for model_name, model in models.items():
            logo = LeaveOneGroupOut()
            patient_true_labels, patient_pred_labels = [], []

            # Hasta Tabanlı Dürüst Çapraz Doğrulama Döngüsü (LOSO)
            for train_idx, test_idx in logo.split(X_scen, y_scen, groups_scen):
                X_train_fold, X_test_fold = X_scen.iloc[train_idx], X_scen.iloc[test_idx]
                y_train_fold, y_test_fold = y_scen.iloc[train_idx], y_scen.iloc[test_idx]

                model.fit(X_train_fold, y_train_fold)
                fold_preds = model.predict(X_test_fold)

                # Hasta pencereleri üzerinde Çoğunluk Oylaması (Majority Voting)
                unique, counts = np.unique(fold_preds, return_counts=True)
                patient_pred_labels.append(unique[np.argmax(counts)])
                patient_true_labels.append(y_test_fold.values[0])

            acc = accuracy_score(patient_true_labels, patient_pred_labels)
            f1 = f1_score(patient_true_labels, patient_pred_labels, average='macro')

            master_results.append({
                "Klinik Senaryo": title,
                "Model Algoritması": model_name,
                "Özellik Yaklaşımı": setup_name,
                "LOSO Accuracy": acc,
                "LOSO Macro F1": f1
            })

            # Sadece Şampiyon Kombinasyonun (LightGBM + Füzyon) matrisini sakla
            if "LightGBM" in model_name and "Füzyon" in setup_name:
                cm = confusion_matrix(patient_true_labels, patient_pred_labels)
                champion_confusion_matrices[title] = (cm, class_names)

df_master = pd.DataFrame(master_results)
print("\n✔️ Harika! 4 Senaryo x 3 Algoritma x 2 Özellik Kümesi = 24 Büyük Kombinasyon başarıyla tamamlandı!")

In [ ]:
# 5. GELİŞMİŞ METRİKLER, BELİRSİZLİK ANALİZİ VE AKADEMİK KIYASLAMA MOTORU
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Makalenin (Miltiadous et al., 2023) resmi tablolardaki sonuçları [cite: 314, 316]
makale_benchmarks = {
    "Senaryo 1: AD vs Control": {"ACC": 0.7643, "F1": 0.7612},
    "Senaryo 2: FTD vs Control": {"ACC": 0.7243, "F1": 0.6732}
}

for title in df_master['Klinik Senaryo'].unique():
    print(f"\n📊 [ NİHAİ TASARIM MATRİSİ VE AKADEMİK DEĞERLENDİRME - {title} ]")
    print("="*110)

    scen_mask = df_master['Klinik Senaryo'] == title
    sub_df = df_master[scen_mask].copy()

    sub_df = sub_df.sort_values(by="LOSO Macro F1", ascending=False).reset_index(drop=True)
    print(sub_df.to_string(index=False, formatters={'LOSO Accuracy': '{:.2%}'.format, 'LOSO Macro F1': '{:.2%}'.format}))
    print("-"*110)

    # Şampiyon (LightGBM + Füzyon) ve Bizim İç Baseline (LightGBM + PSD) değerlerini çek
    lgbm_fusion = sub_df[(sub_df['Model Algoritması'] == 'LightGBM (Modern Boosting)') & (sub_df['Özellik Yaklaşımı'] == 'Önerilen Çoklu Füzyon (PSD+CWT+EMD)')]
    lgbm_psd = sub_df[(sub_df['Model Algoritması'] == 'LightGBM (Modern Boosting)') & (sub_df['Özellik Yaklaşımı'] == 'Klasik Baseline (Sadece PSD)')]

    # En iyi dürüst skoru (LightGBM tabanlı) baz alarak belirsizlik hesapla
    en_iyi_acc = sub_df['LOSO Accuracy'].max() if lgbm_fusion.empty else lgbm_fusion['LOSO Accuracy'].values[0]
    en_iyi_f1 = sub_df['LOSO Macro F1'].max() if lgbm_fusion.empty else lgbm_fusion['LOSO Macro F1'].values[0]

    std_err_f1 = np.sqrt((en_iyi_f1 * (1 - en_iyi_f1)) / 88)
    std_err_acc = np.sqrt((en_iyi_acc * (1 - en_iyi_acc)) / 88)

    high_acc_bound = min(1.0, en_iyi_acc + 1.96*std_err_acc)
    high_f1_bound = min(1.0, en_iyi_f1 + 1.96*std_err_f1)

    print(f"🎲 [BONUS - BELİRSİZLİK GÖSTERİMİ]: Şampiyon Kombinasyon için Matematiksel Güven Aralığı (Confidence Interval %95):")
    print(f"   • LOSO Accuracy Aralığı:  %{ (en_iyi_acc - 1.96*std_err_acc)*100:.2f}  ile  %{ high_acc_bound*100:.2f}")
    print(f"   • LOSO Macro F1 Aralığı:  %{ (en_iyi_f1 - 1.96*std_err_f1)*100:.2f}  ile  %{ high_f1_bound*100:.2f}")

    # LİTERATÜR VEYA İÇ BASELINE KIYASLAMA KATMANI
    if title in makale_benchmarks:
        m_acc = makale_benchmarks[title]["ACC"]
        m_f1 = makale_benchmarks[title]["F1"]
        print(f"📖 [LİTERATÜR KIYASLAMA]: Miltiadous et al. (2023) MDPI Makale Benchmark Sonuçları[cite: 314, 316]:")
        print(f"   • Makale Skoru: ACC: {m_acc:.2%}, F1: {m_f1:.2%}")
        print(f"   • Önerilen Sistemimizin Makaleye Göre Net Katkısı (Performance Lift): ACC: {en_iyi_acc - m_acc:+.2%}, F1: {en_iyi_f1 - m_f1:+.2%}")
    else:
        print(f"📖 [LİTERATÜR NOTU]: Orijinal makalede (Miltiadous et al., 2023) bu senaryo için hazır bir benchmark tablosu yoktur.")
        if not lgbm_fusion.empty and not lgbm_psd.empty:
            b_acc = lgbm_psd['LOSO Accuracy'].values[0]
            b_f1 = lgbm_psd['LOSO Macro F1'].values[0]
            print(f"   • Sistemimizin Kendi İç Klasik Baseline (Sadece PSD) Skoru: ACC: {b_acc:.2%}, F1: {b_f1:.2%}")
            print(f"   • Önerilen Çoklu Füzyonun İç Baseline'a Göre Katkısı: ACC: {en_iyi_acc-b_acc:+.2%}, F1: {en_iyi_f1-b_f1:+.2%}")
    print("="*110)

# 4'lü Confusion Matrix Panelini Yeniden Çiz
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
for idx, (t, (cm, class_names)) in enumerate(champion_confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16, 'weight': 'bold'})
    axes[idx].set_title(f"{t}\n(Şampiyon Model: LightGBM + Önerilen Füzyon)", fontsize=12, weight='bold', pad=10)
    axes[idx].set_xlabel("Modelin Koyduğu Teşhis (Predicted)", fontsize=11)
    axes[idx].set_ylabel("Gerçek Klinik Durum (True)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve_dilara(model, X, y, title):
    # Veriyi farklı boyutlarda bölerek başarımı ölçüyoruz
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
    )

    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Eğitim Başarısı', marker='o')
    plt.plot(train_sizes, test_mean, label='Test Başarısı', marker='o')

    plt.title(f"Öğrenme Eğrisi: {title}")
    plt.xlabel("Eğitim Verisi Boyutu")
    plt.ylabel("Skor (F1 Score)")
    plt.legend()
    plt.grid(True)
    plt.show()

# Şampiyon modelin için çizdiriyoruz
# X_set_FUSION ve y_clean senin optimize verilerin
plot_learning_curve_dilara(lgb.LGBMClassifier(max_depth=5, verbosity=-1), X_set_FUSION, y_clean, "LightGBM (Şampiyon Model)")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# LightGBM modeli içinde feature_importances_ değerlerini al
# model_fi, senin LightGBM şampiyon modelin (Eğitilmiş olması lazım)
# Eğer modelin eğitilmiş değilse, burada en iyi skoru veren modeli kullan

importances = model_fi.feature_importances_
feature_names = X_raw.columns # Tüm özniteliklerin isimleri

# İlk 20 en önemli özniteliği al
df_fi = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
df_fi = df_fi.sort_values(by='Importance', ascending=False).head(20)

# Grafik çizimi
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=df_fi, palette='viridis')
plt.title('Şampiyon Modelin (LightGBM) Karar Mekanizması: İlk 20 Öznitelik')
plt.xlabel('Öznitelik Önem Skoru (Gain/Split)')
plt.ylabel('Öznitelik İsimleri')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()